# 02 — Geração de Embeddings (Colab / GPU)

Gera os embeddings dos produtos para cada modelo e grava em `data/embeddings/`:
um parquet em `float32` (`product_id`, `embedding`) e um `.meta.json` com a configuração usada, por modelo.
**Nenhum Elasticsearch é necessário aqui.** A indexação é feita depois, na máquina local, com
`scripts/index_embeddings.py`.

**Entrada:** `data/processed/products_us.parquet` (notebook 01).

**Retomada:** modelos já gerados e compatíveis com o `models.json` são pulados. Se o Colab desconectar,
execute o notebook novamente.

**Depois:** copie a pasta `data/embeddings/` do Drive para a mesma pasta no projeto local.

Para alternar CPU/GPU, altere apenas `DEVICE` na célula de setup.

## 0. Setup

In [ ]:
import sys
from pathlib import Path

# ── Configuração única ───────────────────────────────────────────────
DEVICE = "auto"                     # "cpu" | "cuda" | "auto"
DRIVE_PROJECT_DIR = "MyDrive/tcc"   # pasta do projeto no Google Drive (só Colab)
# ─────────────────────────────────────────────────────────────────────

IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    from google.colab import drive

    drive.mount("/content/drive")
    PROJECT_ROOT = Path("/content/drive") / DRIVE_PROJECT_DIR
    %pip install -q polars pyarrow "sentence-transformers>=2.7" "transformers>=4.51" tqdm
else:
    PROJECT_ROOT = Path.cwd()

assert (PROJECT_ROOT / "src").exists(), f"Pasta 'src' não encontrada em {PROJECT_ROOT}"
sys.path.insert(0, str(PROJECT_ROOT))

from src.device import build_runtime_config, collect

RUNTIME = build_runtime_config(DEVICE)
print(f"Projeto : {PROJECT_ROOT}")
print(f"Colab   : {IN_COLAB}")
print(f"Device  : {RUNTIME.device} (batch_size={RUNTIME.batch_size}, max_seq_length={RUNTIME.max_seq_length})")

In [ ]:
import logging
import time

import polars as pl

from src.embeddings import generate_embeddings
from src.models.registry import list_models
from src.text import prepare_dataframe

logging.basicConfig(level=logging.INFO, format="%(asctime)s [%(levelname)s] %(message)s", datefmt="%H:%M:%S", force=True)

PRODUCTS_PATH = PROJECT_ROOT / "data" / "processed" / "products_us.parquet"
EMBEDDINGS_DIR = PROJECT_ROOT / "data" / "embeddings"

MODELS: list[str] | None = None        # None = todos; ex.: ["bge", "rexbert"]
OVERWRITE = False                      # True refaz mesmo que o arquivo exista e seja compatível
BATCH_SIZE = RUNTIME.batch_size        # reduza se faltar memória da GPU (o BGE-M3 é o mais pesado)
# Limite de tokens por texto durante a geração. None usa o limite de cada modelo (models.json).
# Valores menores aceleram; o valor efetivo fica registrado em <modelo>.meta.json.
MAX_SEQ_LENGTH: int | None = RUNTIME.max_seq_length

assert PRODUCTS_PATH.exists(), f"{PRODUCTS_PATH} não encontrado. Execute o notebook 01 primeiro."

## 1. Catálogo

In [ ]:
products = collect(pl.scan_parquet(PRODUCTS_PATH), RUNTIME.device)
prepared = prepare_dataframe(products)
print(f"Produtos: {prepared.height:,}")
prepared.select("product_id", "product_text").head(3)

## 2. Geração dos embeddings

Um modelo por vez; a memória do modelo é liberada ao final de cada um. Falhas em um modelo não interrompem os demais.

In [ ]:
targets = MODELS or [cfg.alias for cfg in list_models()]
summary: list[dict] = []

for alias in targets:
    started = time.perf_counter()
    try:
        generate_embeddings(
            alias,
            prepared,
            EMBEDDINGS_DIR,
            device=RUNTIME.device,
            batch_size=BATCH_SIZE,
            max_seq_length=MAX_SEQ_LENGTH,
            overwrite=OVERWRITE,
        )
        status, error = "ok", ""
    except Exception as exc:
        logging.getLogger(__name__).error("Falha em %s: %s", alias, exc)
        status, error = "erro", str(exc)
    summary.append({"modelo": alias, "status": status, "segundos": round(time.perf_counter() - started), "erro": error})

pl.DataFrame(summary)

## 3. Verificação dos arquivos gerados

In [ ]:
from src.embeddings import is_complete, read_meta, embeddings_path

rows = []
for alias in targets:
    if not is_complete(EMBEDDINGS_DIR, alias):
        rows.append({"modelo": alias, "completo": False})
        continue
    meta = read_meta(EMBEDDINGS_DIR, alias)
    rows.append(
        {
            "modelo": alias,
            "completo": True,
            "dim": meta.embedding_dim,
            "produtos": meta.n_docs,
            "max_seq_length": meta.max_seq_length,
            "device": meta.device,
            "mb": round(embeddings_path(EMBEDDINGS_DIR, alias).stat().st_size / 1024**2, 1),
        }
    )

report = pl.DataFrame(rows, infer_schema_length=None)
assert report["completo"].all(), "Há modelos sem embeddings completos; veja a tabela da seção 2."
report